In [5]:
# 导入文档加载器和文本切割器
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. 加载你的机密文档
print("正在读取文档...")
loader = Docx2txtLoader("毕业论文.docx")
docs = loader.load()
print(f"✅ 论文加载成功！一共读取了 {len(docs[0].page_content)} 个字符。\n")

# 2. 配置切割策略 (完全复刻 Dify 的高级设置)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,      # 每个碎片最大 800 个字符
    chunk_overlap=150,   # 碎片之间重叠 150 个字符，保留上下文逻辑连贯性
    separators=["\n\n", "\n", "。", "！", "？", "，", " "] # 智能切分规则：优先按段落切，不行再按句号切
)

# 3. 执行物理粉碎
chunks = text_splitter.split_documents(docs)
print(f"🔪 粉碎完毕！你的论文被切成了 {len(chunks)} 个独立的碎片。\n")

# 4. 直观检验：掏出前两个碎片看看长啥样
print("👇 [碎片 1] 内容：")
print("-" * 50)
print(chunks[0].page_content)
print("-" * 50)

print("\n👇 [碎片 2] 内容 (注意看开头是不是和碎片 1 的结尾有重叠)：")
print("-" * 50)
print(chunks[1].page_content)
print("-" * 50)

正在读取文档...
✅ 论文加载成功！一共读取了 17767 个字符。

🔪 粉碎完毕！你的论文被切成了 29 个独立的碎片。

👇 [碎片 1] 内容：
--------------------------------------------------
南昌大学科学技术学院学士学位论文

南昌大学科学技术学院学士学位论文

南昌大学科学技术学院学士学位论文



南昌大学科学技术学院学士学位论文

南昌大学科学技术学院学士学位论文

                                                          密级：         



科学技术学院

  Science and Technology College of Nanchang University

学 士 学 位 论 文

 THESIS  OF  BACHELOR

（2024 — 2026 年）





题   目 基于JavaWeb的本地生活服务信息发布平台的实现与设计





		学 科 部：    信息学科部     系：     计算机系    

专    业： 计算机科学与技术 班级：     2213班    

	      学生姓名：       万涛       学号：   7020822575   

		      指导教师：      占慧敏      职称：   高级工程师   

		起讫日期：        2025年11月—2026年4月        

		



南昌大学 科学技术学院

学士学位论文原创性申明



本人郑重申明：所呈交的论文是本人在导师的指导下独立进行研究所取得的研究成果。除了文中特别加以标注引用的内容外，本论文不包含任何其他个人或集体已经发表或撰写的成果作品。对本文的研究作出重要贡献的个人和集体，均已在文中以明确方式表明。本人完全意识到本申明的法律后果由本人承担。



作者签名：                       日期：







学位论文版权使用授权书
--------------------------------------------------

👇 [碎片 2] 内容 (注意看开头是不是和碎片 1 的结尾有重叠)：
---------------------

In [6]:
# 导入 Embedding 模型和本地向量数据库 Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
import os

# 1. 配置硅基流动的 API 作为我们的“高维空间翻译官”
# 【注意】请把下面这行换成你在硅基流动官网申请的那一串 API 密钥！
siliconflow_api_key = "sk-ghwlqhnnjhncittaotjzrgyomwhftwuzahpkcgjkdngmuuyh" 

print("📡 正在连接硅基流动，呼叫 BAAI/bge-m3 翻译官...")
embeddings = OpenAIEmbeddings(
    openai_api_key=siliconflow_api_key,
    openai_api_base="https://api.siliconflow.cn/v1",
    model="BAAI/bge-m3"
)

# 2. 建立本地向量数据库 (ChromaDB)
print("🧠 正在将 29 个碎片发送给大模型计算多维坐标，并存入本地数据库 (大概需要十几秒)...")

# 核心动作：生成向量并存入数据库！存放路径设为当前目录下的 "my_local_db" 文件夹
vectorstore = Chroma.from_documents(
    documents=chunks, 
    embedding=embeddings,
    persist_directory="./my_local_db" # 数据会持久化保存在你电脑的这个文件夹里
)

print("\n✅ 完美通关！你的本地专属外挂数据库已建好！")
print("看看你左侧的文件树，是不是多了一个叫 'my_local_db' 的文件夹？你的论文现在全是数学魔法了！")

📡 正在连接硅基流动，呼叫 BAAI/bge-m3 翻译官...
🧠 正在将 29 个碎片发送给大模型计算多维坐标，并存入本地数据库 (大概需要十几秒)...

✅ 完美通关！你的本地专属外挂数据库已建好！
看看你左侧的文件树，是不是多了一个叫 'my_local_db' 的文件夹？你的论文现在全是数学魔法了！


In [10]:
# 导入 LangChain 负责组装的工具和调用大模型的组件
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain

# 0. 填入你的通行证 (记得换成你的真实 API 密钥！)
siliconflow_api_key = "sk-ghwlqhnnjhncittaotjzrgyomwhftwuzahpkcgjkdngmuuyh"

# 1. 唤醒你的外接硬盘 (加载刚才建好的本地数据库)
print("💾 正在唤醒本地知识库...")
embeddings = OpenAIEmbeddings(
    openai_api_key=siliconflow_api_key,
    openai_api_base="https://api.siliconflow.cn/v1",
    model="BAAI/bge-m3"
)
# 注意：这里我们设定每次去库里捞取最相关的 3 个碎片 (对应 Dify 里的 Top K = 3)
vectorstore = Chroma(persist_directory="./my_local_db", embedding_function=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 2. 召唤超级大脑 (用硅基流动上非常聪明的 DeepSeek 模型)
print("🧠 正在连接 DeepSeek 大模型...")
llm = ChatOpenAI(
    openai_api_key=siliconflow_api_key,
    openai_api_base="https://api.siliconflow.cn/v1",
    model_name="deepseek-ai/DeepSeek-V3", # 调用深度求索 V3 模型
    temperature=0.1 # 温度越低，回答越严谨，绝不胡说八道
)

# 3. 注入灵魂 (完全复刻我们在 Dify 里的提示词)
system_prompt = (
    "你是一个极其严谨的高级架构师，也是我的毕业答辩指导老师。\n"
    "请你严格基于以下提供的参考资料（Context）来回答问题。\n"
    "如果资料中找不到答案，请直接回答“论文中未涉及此部分”，严禁你自己胡编乱造！\n\n"
    "参考资料：\n{context}"
)

# 组装提示词模板
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# 4. 用链条 (Chain) 把所有零件焊死！
# 这一步相当于 Dify 里把“知识检索”的线连到“LLM”节点上
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

# ==========================================
# 🎯 终极火力测试！在这里输入你想问的问题！
# ==========================================
question = "论文摘要里提到了本项目的什么意义？"
print(f"\n👤 你的问题：{question}")
print("🤖 专属导师正在你的本地数据库中疯狂翻阅，并组织语言...\n")

# 按下发射按钮！
response = rag_chain.invoke({"input": question})

print("👇 [专属导师的完美回答]")
print("=" * 60)
print(response["answer"])
print("=" * 60)

ModuleNotFoundError: No module named 'langchain.chains'